# Reference solutions — open after attempting

Part of the micrograd repetition pack.


These are compact reference implementations. Compare ideas and invariants, not just exact spelling.


In [ ]:
import math

_results = []

def check(name, condition, detail=""):
    ok = bool(condition)
    _results.append(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

def close(a, b, tol=1e-6):
    return abs(a - b) <= tol

def summary():
    print(f"\nScore: {sum(_results)}/{len(_results)} tests passed")


In [ ]:
from math import sin, cos

def f(a,b,c): return -a**3+sin(3*b)-1/c+b**2.5-a**0.5
def analytic_grad(a,b,c): return [-3*a**2-.5*a**-.5,3*cos(3*b)+2.5*b**1.5,c**-2]
def forward_partial(fn,point,argnum,h=1e-6):
    plus=list(point); plus[argnum]+=h
    return (fn(*plus)-fn(*point))/h
def central_partial(fn,point,argnum,h=1e-5):
    plus=list(point); minus=list(point)
    plus[argnum]+=h; minus[argnum]-=h
    return (fn(*plus)-fn(*minus))/(2*h)

print(analytic_grad(2,3,4))


In [ ]:
from math import exp, log, sin, cos

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f'**{power}')
        def _backward():
            self.grad += power * self.data ** (power - 1) * out.grad
        out._backward = _backward
        return out

    def sin(self):
        out = Value(sin(self.data), (self,), 'sin')
        def _backward():
            self.grad += cos(self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        e2x = (2 * self).exp()
        return (e2x - 1) / (e2x + 1)

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __rtruediv__(self, other): return Value(other) * self ** -1

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [ ]:
def trace(root):
    nodes,edges=set(),set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v)); build(child)
    build(root)
    return nodes,edges

def topological(root):
    topo,visited=[],set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev: build(child)
            topo.append(v)
    build(root)
    return topo

def backward(root):
    root.grad=1.0
    for node in reversed(topological(root)): node._backward()


In [ ]:
def softmax(logits):
    counts=[x.exp() for x in logits]
    total=sum(counts)
    return [x/total for x in counts]

def stable_softmax(logits):
    shift=max(x.data for x in logits)
    counts=[(x-shift).exp() for x in logits]
    total=sum(counts)
    return [x/total for x in counts]

def nll_loss(logits,target): return -softmax(logits)[target].log()


In [ ]:
import random

class Module:
    def parameters(self): return []
    def zero_grad(self):
        for p in self.parameters(): p.grad = 0.0

class Neuron(Module):
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    def __call__(self, x):
        if len(x) != len(self.w):
            raise ValueError(f"expected {len(self.w)} inputs, got {len(x)}")
        return (sum(wi * xi for wi, xi in zip(self.w, x)) + self.b).tanh()
    def parameters(self): return self.w + [self.b]

class Layer(Module):
    def __init__(self, nin, nout): self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, nin, nouts):
        sizes = [nin] + list(nouts)
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for layer in self.layers for p in layer.parameters()]


In [ ]:
def squared_error(model,xs,ys):
    predictions=[model(x) for x in xs]
    return sum((yp-y)**2 for yp,y in zip(predictions,ys)),predictions

def train_step(model,xs,ys,learning_rate=.01):
    loss,_=squared_error(model,xs,ys)
    model.zero_grad(); loss.backward()
    for p in model.parameters(): p.data -= learning_rate*p.grad
    return loss.data

def train(model,xs,ys,steps=50,learning_rate=.05):
    return [train_step(model,xs,ys,learning_rate) for _ in range(steps)]

def torch_gradient_check():
    import torch
    a=torch.tensor(2.,requires_grad=True); b=torch.tensor(3.,requires_grad=True); c=torch.tensor(4.,requires_grad=True)
    L=-a**3+torch.sin(3*b)-1/c+b**2.5-a**.5
    L.backward()
    return [a.grad.item(),b.grad.item(),c.grad.item()]


In [ ]:
# End-to-end smoke test
a,b,c=Value(2),Value(3),Value(4)
L=-a**3+(3*b).sin()-1/c+b**2.5-a**0.5
L.backward()
print("gradients:",[a.grad,b.grad,c.grad])

logits=[Value(0),Value(3),Value(-2),Value(1)]
loss=nll_loss(logits,3); loss.backward()
print("NLL:",loss.data)
print("logit grads:",[x.grad for x in logits])

random.seed(1337)
model=MLP(3,[4,4,1])
xs=[[2.,3.,-1.],[3.,-1.,.5],[.5,1.,1.],[1.,1.,-1.]]; ys=[1.,-1.,-1.,1.]
losses=train(model,xs,ys)
print("training loss:",losses[0],"->",losses[-1])
